# Hybrid Quantum GNNs for Molecular Toxicity Classification
**BMS Institute of Technology and Management** | ML – BCS602 | AY 2025–2026  
**Guide:** Dr. Nagabhushan SV

| USN | Name |
|---|---|
| 1BY23CS014 | Aishwarya J A |
| 1BY23CS026 | Anurag Rai |
| 1BY23CS053 | Dasiga Venkata Ashish Kumar |
| 1BY23CS068 | G Nithish |

## 1. Install Dependencies

In [1]:
!pip install torch torch-geometric rdkit pennylane scikit-learn pandas tqdm matplotlib seaborn requests -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 22.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 82.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 82.1 MB/s eta 0:00:00:00:0100:01


## 2. Clone Repository (Colab only)

In [2]:
try:
    import google.colab
    try:
        !rm -rf /content/HQGNN
    except:
        pass
    !git clone https://github.com/r-anurag/Hybrid-Quantum-GNNs-for-Molecular-Toxicity-Classification.git /content/HQGNN
    %cd /content/HQGNN
    !ls -la
    print('Repository cloned successfully')
except:
    print('Running locally, skipping clone')

Cloning into '/content/HQGNN'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 143 (delta 81), reused 101 (delta 39), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 74.91 KiB | 1.19 MiB/s, done.
Resolving deltas: 100% (81/81), done.
/content/HQGNN
total 64
drwxr-xr-x 4 root root  4096 Apr 27 10:15 .
drwxr-xr-x 1 root root  4096 Apr 27 10:15 ..
drwxr-xr-x 8 root root  4096 Apr 27 10:15 .git
-rw-r--r-- 1 root root   531 Apr 27 10:15 .gitignore
-rw-r--r-- 1 root root 15346 Apr 27 10:15 notebook_fixed.ipynb
-rw-r--r-- 1 root root  8392 Apr 27 10:15 PROJECT_PLAN.md
-rw-r--r-- 1 root root  2936 Apr 27 10:15 README.md
-rw-r--r-- 1 root root    96 Apr 27 10:15 requirements.txt
drwxr-xr-x 3 root root  4096 Apr 27 10:15 src
-rw-r--r-- 1 root root  1919 Apr 27 10:15 test_edge_vqc_smoke.py
-rw-r--r-- 1 root root   957 Apr 27 10:15 test_vqc_shapes.py
Repository cloned successfully


: 

## 3. Clear Module Cache

In [3]:
import sys

modules_to_remove = []
for key in list(sys.modules.keys()):
    if any(x in key for x in ['models', 'data_pipeline', 'train', 'evaluate']):
        modules_to_remove.append(key)

for module in modules_to_remove:
    del sys.modules[module]

print(f"Cleared {len(modules_to_remove)} cached modules")
print("Ready to import fresh code from GitHub")

Cleared 1 cached modules
Ready to import fresh code from GitHub


: 

## 4. Setup Environment

In [4]:
import os
import warnings
warnings.filterwarnings("ignore")

try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

if os.path.exists('src'):
    os.chdir('src')
elif os.path.exists('../src'):
    os.chdir('../src')
else:
    print('ERROR: src/ folder not found')
    print(f'Current directory: {os.getcwd()}')
    print(f'Contents: {os.listdir()}')

print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"Working directory: {os.getcwd()}")

Environment: Colab
Working directory: /content/HQGNN/src


: 

## 5. Verify Fixed Code

In [5]:
with open('models/hybrid_qgnn.py', 'r') as f:
    content = f.read()

checks = {
    "Quantum scaling parameter": "self.quantum_scale = nn.Parameter" in content,
    "Input splitting in edge circuit": "node_inputs = inputs[..., :n_qubits]" in content,
    "Improved circuit (RZ gates)": "qml.RZ(weights[layer, i, 1]" in content,
    "Learnable input scale": "self.input_scale = nn.Parameter" in content,
}

print("Code Verification:")
all_passed = True
for name, passed in checks.items():
    status = "✓" if passed else "✗"
    print(f"  {status} {name}")
    all_passed = all_passed and passed

if all_passed:
    print("\n✓ All fixes verified! Using correct code from GitHub.")
else:
    print("\n✗ WARNING: Some fixes missing!")

Code Verification:
  ✓ Quantum scaling parameter
  ✓ Input splitting in edge circuit
  ✓ Improved circuit (RZ gates)
  ✓ Learnable input scale

✓ All fixes verified! Using correct code from GitHub.


: 

## 6. Import Libraries

In [6]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.10.0+cu128
Device: cuda
GPU: Tesla T4


: 

## 7. Setup Checkpoint Directory

In [7]:
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        CHECKPOINT_DIR = '/content/drive/MyDrive/HQGNN_checkpoints'
        print(f"Checkpoints: {CHECKPOINT_DIR}")
    except:
        CHECKPOINT_DIR = None
        print("No checkpoint directory")
else:
    CHECKPOINT_DIR = '../checkpoints'
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"Checkpoints: {CHECKPOINT_DIR}")

Mounted at /content/drive
Checkpoints: /content/drive/MyDrive/HQGNN_checkpoints


: 

## 8. Load Datasets

In [8]:
from data_pipeline import load_dataset

print("Loading datasets...\n")
datasets = {}

for name in ("clintox", "tox21"):
    print(f"Loading {name.upper()}...")
    data_list, class_weights, tasks = load_dataset(name)
    datasets[name] = (data_list, class_weights, tasks)
    print(f"  {len(data_list)} molecules, {len(tasks)} tasks\n")

Loading datasets...

Loading CLINTOX...
✓ Downloaded to /root/.cache/hqgnn/clintox.csv.gz


[10:16:35] Explicit valence for atom # 0 N, 4, is greater than permitted
[10:16:36] Can't kekulize mol.  Unkekulized atoms: 9
[10:16:37] Can't kekulize mol.  Unkekulized atoms: 4
[10:16:37] Can't kekulize mol.  Unkekulized atoms: 4


⚠ Skipped 5/1484 invalid SMILES
  1479 molecules, 2 tasks

Loading TOX21...
✓ Downloaded to /root/.cache/hqgnn/tox21.csv.gz


[10:16:38] WARNING: not removing hydrogen atom without neighbors
[10:16:39] Explicit valence for atom # 8 Al, 6, is greater than permitted
[10:16:40] Explicit valence for atom # 3 Al, 6, is greater than permitted
[10:16:40] Explicit valence for atom # 4 Al, 6, is greater than permitted
[10:16:42] Explicit valence for atom # 4 Al, 6, is greater than permitted
[10:16:43] Explicit valence for atom # 9 Al, 6, is greater than permitted
[10:16:43] Explicit valence for atom # 5 Al, 6, is greater than permitted
[10:16:44] Explicit valence for atom # 16 Al, 6, is greater than permitted
[10:16:45] Explicit valence for atom # 20 Al, 6, is greater than permitted


⚠ Skipped 27/7831 invalid SMILES
  7804 molecules, 12 tasks



: 

## 9. Define Model Variants (Edge Embedding First)

In [9]:
# Clear models cache RIGHT BEFORE importing
import sys
for key in list(sys.modules.keys()):
    if 'models' in key:
        del sys.modules[key]
print('Cleared models cache before import\n')

# Import with fresh circuits
from models import GCN, HybridQGNN, QuantumOnly

IN_CHANNELS = 10
EPOCHS = 100
LR = 1e-3
BATCH = 64
N_FOLDS = 5
EARLY_STOP_PATIENCE = 20
WEIGHT_DECAY = 1e-4

def get_variants(num_tasks, ds_name):
    if ds_name == 'clintox':
        # All CLINTOX experiments already completed
        return {}
    elif ds_name == 'tox21':
        # Only run the remaining TOX21 models
        return {
            "Hybrid 4-qubit": lambda: HybridQGNN(
                IN_CHANNELS, gcn_hidden=64, gcn_embed=32,
                n_qubits=4, n_layers=2, num_tasks=num_tasks
            ),
            "Hybrid 8-qubit": lambda: HybridQGNN(
                IN_CHANNELS, gcn_hidden=64, gcn_embed=32,
                n_qubits=8, n_layers=2, num_tasks=num_tasks
            ),
        }
    else:
        # Default for any other dataset
        return {
            "Hybrid 4-qubit + Edge": lambda: HybridQGNN(
                IN_CHANNELS, gcn_hidden=64, gcn_embed=32,
                n_qubits=4, n_layers=2, num_tasks=num_tasks, edge_embed=True
            ),
            "Classical GCN": lambda: GCN(
                IN_CHANNELS, hidden=64, embed_dim=32, num_tasks=num_tasks
            ),
            "Quantum-only 4-qubit": lambda: QuantumOnly(
                IN_CHANNELS, n_qubits=4, n_layers=2, num_tasks=num_tasks
            ),
            "Hybrid 4-qubit": lambda: HybridQGNN(
                IN_CHANNELS, gcn_hidden=64, gcn_embed=32,
                n_qubits=4, n_layers=2, num_tasks=num_tasks
            ),
            "Hybrid 8-qubit": lambda: HybridQGNN(
                IN_CHANNELS, gcn_hidden=64, gcn_embed=32,
                n_qubits=8, n_layers=2, num_tasks=num_tasks
            ),
        }

print(f"Configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Early stop patience: {EARLY_STOP_PATIENCE}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"  Metric: ROC-AUC")
print(f"  Device: {DEVICE}")

# Test edge embedding model with forward pass
print("\nTesting edge embedding model...")
try:
    test_model = HybridQGNN(10, n_qubits=4, n_layers=2, num_tasks=2, edge_embed=True)
    print("✓ Model created")
    
    # Test forward pass with real data
    sample_data = datasets['clintox'][0][0]
    test_model.eval()
    with torch.no_grad():
        output = test_model(sample_data)
    print(f"✓ Forward pass successful, output shape: {output.shape}")
    del test_model
except Exception as e:
    print(f"✗ Test failed: {e}")
    import traceback
    traceback.print_exc()
    raise

Cleared models cache before import

Configuration:
  Epochs: 100
  Early stop patience: 20
  Weight decay: 0.0001
  Metric: ROC-AUC
  Device: cuda

Testing edge embedding model...
✓ Model created
✓ Forward pass successful, output shape: torch.Size([1, 2])


: 

## 10. Run Experiments

In [10]:
from evaluate import cross_validate
import time
import pickle
import os

intermediate_file = "../results/intermediate_results.pkl"

# Load intermediate results if exist
if os.path.exists(intermediate_file):
    with open(intermediate_file, 'rb') as f:
        all_rows = pickle.load(f)
    print(f"Loaded {len(all_rows)} intermediate results")
    done = set((r['Model'], r['Dataset']) for r in all_rows)
else:
    all_rows = []
    done = set()

# Load existing results from CSV if available
if os.path.exists("../results/results.csv"):
    existing_df = pd.read_csv("../results/results.csv")
    new_rows = 0
    for _, row in existing_df.iterrows():
        key = (row['Model'], row['Dataset'])
        if key in done:
            continue
        all_rows.append({
            "Model": row['Model'],
            "Dataset": row['Dataset'],
            "Params": int(row['Params']),
            "auc_mean": row['auc_mean'],
            "auc_std": row['auc_std'],
            "time_mean": row['time_mean'],
            "epochs_mean": row['epochs_mean'],
            "epochs_std": row['epochs_std'],
            "histories": []  # Not available from CSV
        })
        done.add(key)
        new_rows += 1
    print(f"Loaded {len(existing_df)} existing results from CSV ({new_rows} new)")
    if new_rows > 0 or not os.path.exists(intermediate_file):
        os.makedirs(os.path.dirname(intermediate_file), exist_ok=True)
        with open(intermediate_file, 'wb') as f:
            pickle.dump(all_rows, f)
        print(f"Saved combined results to intermediate cache ({len(all_rows)} rows)")

start_time = time.time()

for ds_name, (data_list, class_weights, tasks) in datasets.items():
    num_tasks = len(tasks)
    print(f"\n{'='*70}")
    print(f"Dataset: {ds_name.upper()}  ({len(data_list)} molecules, {num_tasks} tasks)")
    print(f"{'='*70}")

    for name, model_fn in get_variants(num_tasks, ds_name).items():
        if (name, ds_name) in done:
            print(f"\n>> {name} (SKIPPED - already done)")
            continue
            
        print(f"\n>> {name}")
        
        model_temp = model_fn()
        n_params = sum(p.numel() for p in model_temp.parameters() if p.requires_grad)
        del model_temp
        print(f"   Parameters: {n_params:,}")

        results = cross_validate(
            model_fn, data_list,
            n_splits=N_FOLDS, epochs=EPOCHS, lr=LR,
            batch_size=BATCH, device=DEVICE, verbose=False,
            checkpoint_dir=CHECKPOINT_DIR,
            model_name=name.replace(" ", "_"),
            dataset_name=ds_name,
            early_stop_patience=EARLY_STOP_PATIENCE,
            weight_decay=WEIGHT_DECAY
        )
        
        row = {
            "Model": name,
            "Dataset": ds_name,
            "Params": n_params,
            "auc_mean": results['auc_mean'],
            "auc_std": results['auc_std'],
            "time_mean": results['time_mean'],
            "epochs_mean": results['epochs_mean'],
            "epochs_std": results['epochs_std'],
            "histories": results['histories']
        }
        all_rows.append(row)
        
        # Save intermediate results
        with open(intermediate_file, 'wb') as f:
            pickle.dump(all_rows, f)
        
        print(f"   AUC: {results['auc_mean']:.4f} ± {results['auc_std']:.4f}")
        print(f"   Epochs: {results['epochs_mean']:.1f} ± {results['epochs_std']:.1f}")
        print(f"   Time/epoch: {results['time_mean']:.2f}s")

elapsed = time.time() - start_time
print(f"\n{'='*70}")
print(f"Total time: {elapsed/60:.1f} minutes")
print(f"{'='*70}")

df = pd.DataFrame([{k: v for k, v in r.items() if k != "histories"} for r in all_rows])
os.makedirs("../results", exist_ok=True)
df.to_csv("../results/results.csv", index=False)
print("\nResults saved to ../results/results.csv")

append_file = "../results/1stresults.csv"
append_header = not os.path.exists(append_file)
df.to_csv(append_file, mode='a', header=append_header, index=False)
print(f"Results appended to {append_file}")

# Clean up intermediate file
if os.path.exists(intermediate_file):
    os.remove(intermediate_file)


Dataset: CLINTOX  (1479 molecules, 2 tasks)

Dataset: TOX21  (7804 molecules, 12 tasks)

>> Quantum-only 4-qubit
   Parameters: 1,250

-- Fold 1/5 --
  ROC-AUC=0.6855  epochs=51  time/epoch=5.15s

-- Fold 2/5 --
  ROC-AUC=0.7300  epochs=100  time/epoch=5.05s

-- Fold 3/5 --
  ROC-AUC=0.7382  epochs=96  time/epoch=5.00s

-- Fold 4/5 --
  ROC-AUC=0.7087  epochs=57  time/epoch=5.04s

-- Fold 5/5 --
  ROC-AUC=0.7489  epochs=100  time/epoch=5.01s


FileNotFoundError: [Errno 2] No such file or directory: '../results/intermediate_results.pkl'

: 

## 11. Results Table

In [ ]:
print(df[['Dataset', 'Model', 'auc_mean', 'auc_std', 'epochs_mean']].to_string(index=False))

NameError: name 'df' is not defined

: 

## 12. Performance Comparison

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

for i, ds_name in enumerate(["clintox", "tox21"]):
    sub = df[df["Dataset"] == ds_name]
    x = np.arange(len(sub)) + i * 0.35
    ax.bar(x, sub["auc_mean"], width=0.35, yerr=sub["auc_std"],
           capsize=4, label=ds_name.upper(), alpha=0.8)

ax.set_xticks(np.arange(len(sub)) + 0.175)
ax.set_xticklabels(sub["Model"], rotation=45, ha='right')
ax.set_title("ROC-AUC Comparison", fontsize=14, fontweight='bold')
ax.set_ylabel("ROC-AUC", fontsize=12)
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../results/performance_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

: 

## 13. Training Curves

In [ ]:
n_plots = min(len(all_rows), 10)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

def pad_history(values, length):
    return values + [np.nan] * (length - len(values))

for idx, row in enumerate(all_rows[:n_plots]):
    ax = axes[idx]
    histories = row["histories"]
    max_len = max(len(h["train_loss"]) for h in histories)
    train_losses = np.nanmean([pad_history(h["train_loss"], max_len) for h in histories], axis=0)
    val_losses = np.nanmean([pad_history(h["val_loss"], max_len) for h in histories], axis=0)
    
    ax.plot(train_losses, label="Train", linewidth=2, alpha=0.8)
    ax.plot(val_losses, label="Val", linewidth=2, alpha=0.8)
    ax.set_title(f"{row['Model']}\n{row['Dataset'].upper()}", fontsize=10)
    ax.set_xlabel("Epoch", fontsize=9)
    ax.set_ylabel("Loss", fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

for idx in range(n_plots, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig("../results/training_curves.png", dpi=150, bbox_inches='tight')
plt.show()

: 